# Compare Infomap and Louvain with NetworkX

This tutorial shows how to run Infomap next to Louvain community detection in a NetworkX workflow. It uses Zachary's karate club graph because it is small, built into NetworkX, and common in community-detection examples.

Infomap is registered as a [NetworkX backend](https://networkx.org/documentation/stable/reference/backends.html), so it can be called through the same `nx.community` API as Louvain: `nx.community.infomap_communities(G, backend="infomap")`. Infomap optimizes the map equation (it searches for a partition that compresses the description of flow on the network); Louvain optimizes modularity. The goal here is not to declare a universal winner, but to make the outputs easy to run, inspect, compare, visualize, and export.


In [ ]:
from pathlib import Path

import infomap
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

print("infomap:", infomap.__version__)
print("networkx:", nx.__version__)
print("pandas:", pd.__version__)


## Load a small graph

`nx.karate_club_graph()` is undirected and unweighted. For weighted graphs, pass the edge attribute name with `weight="weight"`; for unweighted graphs, pass `weight=None`. Infomap detects directed NetworkX graphs automatically when the input is a `DiGraph`.


In [ ]:
graph = nx.karate_club_graph()
nodes = list(graph.nodes())
print(graph)
print("nodes:", graph.number_of_nodes())
print("edges:", graph.number_of_edges())


## Run the community methods

Both methods are called through NetworkX's `nx.community` API and return a partition as `list[set]` with the original node labels preserved. Louvain is built into NetworkX; Infomap dispatches to its backend via `backend="infomap"`. Extra Infomap options (e.g. `num_trials`, `two_level`, `markov_time`) are passed as backend-only keyword arguments.

`nx.community.infomap_communities` requires a NetworkX release that defines it. On older versions, the cell below falls back to calling the `infomap` package directly, which gives an identical result.


In [ ]:
def partition_to_labels(partition, ordered_nodes):
    labels = {}
    for community_id, community in enumerate(partition, start=1):
        for node in community:
            labels[node] = community_id
    return [labels[node] for node in ordered_nodes]


# Infomap through the NetworkX backend dispatch API, with a fallback for
# NetworkX versions that do not yet provide infomap_communities.
if hasattr(nx.community, "infomap_communities"):
    infomap_partition = nx.community.infomap_communities(
        graph,
        weight=None,
        seed=123,
        num_trials=20,
        backend="infomap",
    )
else:
    infomap_partition = infomap.find_communities(
        graph,
        weight=None,
        seed=123,
        num_trials=20,
    )

louvain_partition = nx.community.louvain_communities(
    graph,
    weight=None,
    seed=123,
)

print("Infomap communities:", len(infomap_partition))
print("Louvain communities:", len(louvain_partition))


## Compare assignments

Community IDs are local labels. Their numeric values only identify groups within one result; they should not be interpreted as stable or ordered labels across methods.


In [ ]:
df = pd.DataFrame(
    {
        "node": nodes,
        "club": [graph.nodes[node]["club"] for node in nodes],
        "infomap": partition_to_labels(infomap_partition, nodes),
        "louvain": partition_to_labels(louvain_partition, nodes),
    }
)

df.head(10)


In [ ]:
summary = df.drop(columns="node").nunique().rename("communities").to_frame()
summary


## Simple similarity metrics

Adjusted mutual information (AMI) and normalized mutual information (NMI) compare each detected assignment with the known karate-club split. They are useful checks, but they do not replace inspecting the graph and understanding the objective each method optimizes.


In [ ]:
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score

metrics = pd.DataFrame(
    [
        {
            "method": "infomap",
            "AMI vs truth": adjusted_mutual_info_score(df["club"], df["infomap"]),
            "NMI vs truth": normalized_mutual_info_score(df["club"], df["infomap"]),
        },
        {
            "method": "louvain",
            "AMI vs truth": adjusted_mutual_info_score(df["club"], df["louvain"]),
            "NMI vs truth": normalized_mutual_info_score(df["club"], df["louvain"]),
        },
    ]
)
metrics


## Visualize the partitions

The same layout is reused for each method so the visual comparison focuses on module assignments rather than node placement.


In [ ]:
methods = ["infomap", "louvain"]
pos = nx.spring_layout(graph, seed=123)
fig, axes = plt.subplots(1, len(methods), figsize=(5 * len(methods), 4), squeeze=False)

for ax, method in zip(axes[0], methods):
    colors = [df.loc[df["node"] == node, method].iloc[0] for node in graph.nodes]
    nx.draw_networkx(
        graph,
        pos=pos,
        node_color=colors,
        cmap="tab20",
        with_labels=True,
        node_size=450,
        edge_color="#999999",
        ax=ax,
    )
    ax.set_title(method.title())
    ax.axis("off")

plt.tight_layout()


## Export Infomap results

The backend dispatch API returns just the partition. When you also want to annotate the graph (write module IDs and flow back to nodes, optionally with hierarchical attributes), call `infomap.find_communities` directly with `module_attribute` / `flow_attribute`, then export with NetworkX's GraphML or GEXF writers.


In [ ]:
export_graph = graph.copy()
infomap.find_communities(
    export_graph,
    weight=None,
    module_attribute="infomap",
    flow_attribute="infomap_flow",
    seed=123,
    num_trials=20,
)

export_dir = Path("output")
export_dir.mkdir(exist_ok=True)
graphml_path = export_dir / "karate-infomap.graphml"
gexf_path = export_dir / "karate-infomap.gexf"

nx.write_graphml(export_graph, graphml_path)
nx.write_gexf(export_graph, gexf_path)
print("wrote:", graphml_path)
print("wrote:", gexf_path)

pd.DataFrame.from_dict(dict(export_graph.nodes(data=True)), orient="index").head()


## Citation

If you use Infomap in published work, cite the Infomap software and the map equation literature. See the citation information in the repository and the Infomap user guide for the current recommended references.
